# DynaSurvCausalOnline Model Validation
This notebook implements the validation checks specified in `latex/model_validation.tex`.

## Objectives
1. **Factual Consistency**: Calibration and discriminative performance (C-index, Brier Score).
2. **Counterfactual Stability**: Pairwise constraints and temporal coherence.
3. **Clinical Plausibility**: Stratified analysis by biomarkers.
4. **Policy Evaluation**: Regret analysis.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.nonparametric import CensoringDistributionEstimator
from sksurv.util import Surv
from sksurv.functions import StepFunction

from torchsurv.metrics.brier_score import BrierScore
from torchsurv.metrics.cindex import ConcordanceIndex
from torchsurv.stats.ipcw import get_ipcw

from CausalSurv.model.DynaSurvCausalOnline import DynaSurvCausalOnline
from CausalSurv.data.CV_online_data_utils import ESMEOnlineDataModuleCV

from lifelines import KaplanMeierFitter

from lifelines.fitters.kaplan_meier_fitter import KaplanMeierFitter
from lifelines.statistics import logrank_test

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Update these paths as needed or load from a central config
MODEL_PATH = "/Users/malek/TheLAB/DynaSurv/models/HR+HER2-/4lines/seed_42/checkpoints/H100_RES1/dynaSurvCausalOnline-bestIBS-epoch=34-average_ibs= 0.1486.ckpt"
# MODEL_PATH = "/Users/malek/TheLAB/DynaSurv/models/HR+HER2-/4lines/seed_42/checkpoints/dynaSurvCausalOnline-bestIBS-epoch=04-average_ibs= 0.2147.ckpt"
CONFIG_PATH = "/Users/malek/TheLAB/DynaSurv/configs/config_train.toml"
SPLIT_SEED = 1769945024
SUBTYPE = "HR+HER2-"
HORIZON = 100
N_LINES = 4
BATCH_SIZE = 256

##  Load Model and Data

In [ ]:
# Load Model
print(f"Loading model from {MODEL_PATH}...")
model = DynaSurvCausalOnline.load_from_checkpoint(
    MODEL_PATH, map_location=torch.device("cpu")
)
model.eval()
model.freeze()
print("Model loaded.")

# Load Data
print("Loading data...")
data_module = ESMEOnlineDataModuleCV(
    data_dir="../data/",
    subtype=SUBTYPE,
    n_intervals=len(model.interval_bounds) - 1,
    n_lines=N_LINES,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    final_training=True,
    num_workers=4,
    split_seed=SPLIT_SEED,
)
data_module.prepare_data()
data_module.setup()
print("Data loaded.")


## Compute Censoring Distribution (KM)

In [ ]:
train_loader = data_module.train_dataloader()

train_times = [[] for _ in range(N_LINES)]
train_events = [[] for _ in range(N_LINES)]

print("Accumulating training data for KM estimator...")
for batch in tqdm(train_loader, desc="Training Data"):
    XPd, X_static, interval_idx, treatment_indices, time, event, mask, patient_id = (
        batch
    )
    for line in range(N_LINES):
        valid = mask[:, line].bool()
        if valid.any():
            t_batch = time[valid, line].numpy()
            e_batch = event[valid, line].bool().numpy()
            train_times[line].extend(t_batch)
            train_events[line].extend(e_batch)



In [ ]:
print(data_module.n_intervals)

## Factual Consistency Check

In [ ]:
test_loader = data_module.test_dataloader()
interval_bounds = model.interval_bounds.to("cpu")

LINE_WINDOWS = torch.tensor([100., 75., 50., 30.])  # days since onset

# harell_c_indexes = {}
# uno_c_indexes = {}
antolini_c_indexes = {}
integrated_brier_scores = {}
brier_scores = {line_idx: {'times': None, 'scores': None} for line_idx in range(N_LINES)}

all_predictions = []  # Store for later analysis

print("Running inference on Test set...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch

        discrete_hazards = torch.sigmoid(
            model.get_factual_predictions(XPd, X_static, treatment_indices)[0]
        )  # (batch, n_lines, n_intervals)

        discrete_survival = torch.cumprod(1 - discrete_hazards, dim=2)
        discrete_survival = torch.cat(
            [torch.ones_like(discrete_survival[:, :, :1]), discrete_survival], dim=2
        )  # (batch, n_lines, n_intervals + 1) to account for S(0)=1

        discrete_cumhazards = torch.cumsum(discrete_hazards, dim=2)
        discrete_cumhazards = torch.cat(
            [torch.zeros_like(discrete_cumhazards[:, :, :1]), discrete_cumhazards],
            dim=2,
        )  # (batch, n_lines, n_intervals + 1) to account for H(0)=0


        # Loop over lines to compute metrics
        for line_idx in range(N_LINES):
            valid_mask = mask[:, line_idx].bool()
            if not valid_mask.any():
                continue

            t_line = time[valid_mask, line_idx].cpu()
            e_line = event[valid_mask, line_idx].bool().cpu()
            line_discrete_survival = discrete_survival[
                valid_mask, line_idx, :
            ]
            line_discrete_cumhazards = discrete_cumhazards[
                valid_mask, line_idx, :
            ]

            c_index_td, _ = model.eval_cindex_ipcw(
                train_events=torch.tensor(
                    train_events[line_idx], dtype=torch.bool, device="cpu"
                ),
                train_times=torch.tensor(
                    train_times[line_idx], dtype=torch.float32, device="cpu"
                ),
                test_events=e_line.cpu(),
                test_times=t_line.cpu(),
                discrete_cumhazards=line_discrete_cumhazards.cpu(),
                interval_bounds=interval_bounds[line_idx].cpu(),
                device="cpu",
            )
            antolini_c_indexes[line_idx] = c_index_td

            ibs_line, bs_line, _, bs_eval_times = model.eval_brier_score_ipcw(
                train_events=torch.tensor(
                    train_events[line_idx], dtype=torch.bool, device="cpu"
                ),
                train_times=torch.tensor(
                    train_times[line_idx], dtype=torch.float32, device="cpu"
                ),
                test_events=e_line.cpu(),
                test_times=t_line.cpu(),
                discrete_survival=line_discrete_survival.cpu(),
                tmax = LINE_WINDOWS[line_idx].item(),
                interval_bounds=interval_bounds[line_idx].cpu(),
                device="cpu",
            )
            integrated_brier_scores[line_idx] = ibs_line
            brier_scores[line_idx]['scores'] = bs_line.detach().cpu().numpy()
            brier_scores[line_idx]['times'] = bs_eval_times.detach().cpu().numpy()

        # Store predictions for later plots
        all_predictions.append(
            {
                "patient_id": patient_id.cpu().numpy(),
                "mask": mask.cpu().numpy(),
                "time": time.cpu().numpy(),
                "event": event.cpu().numpy(),
                "predicted_survival": discrete_survival.cpu().numpy(),
                "predicted_cumhazard": discrete_cumhazards.cpu().numpy(),
                "treatment": treatment_indices.cpu().numpy(),
                "XPd": XPd.cpu().numpy(),
                "X_static": [t.cpu().numpy() for t in X_static],
            }
        )

# Aggregate Metrics
print("\n--- Factual Consistency Metrics ---")
results_df = []
for line in range(N_LINES):
    Antolini_CI = antolini_c_indexes[line]
    ibs = integrated_brier_scores[line]
    print(
        f"Line {line + 1}: Antolini C-Index = {Antolini_CI:.4f}, IBS = {ibs:.4f}"
    )
    results_df.append({"Line": line + 1, "Metric": "C-Index", "Value": Antolini_CI})
    results_df.append({"Line": line + 1, "Metric": "IBS", "Value": ibs})

fig, ax = plt.subplots(nrows=1, ncols=N_LINES, figsize=(5 * N_LINES, 4))
for line in range(N_LINES):
    ax[line].plot(
        brier_scores[line]["times"],
        brier_scores[line]["scores"],
        label=f"Line {line + 1}",
    )
    ax[line].set_title(f"Brier Score - Line {line + 1}")
    ax[line].set_xlabel("Time")
    ax[line].set_ylabel("Brier Score")
    ax[line].legend()
plt.show()

## Calibration Plot

In [ ]:
def plot_calibration(line_idx, time_point_idx=3, n_bins=30):
    # Gather data
    preds_surv = []
    obs_times = []
    obs_events = []
    time_val = model.interval_bounds[line_idx][time_point_idx].item()

    for batch_res in all_predictions:
        mask_line = batch_res["mask"][:, line_idx].astype(bool)
        if not mask_line.any():
            print("No valid data for this line in the batch.")
            continue

        preds_surv.extend(batch_res["predicted_survival"][mask_line, line_idx, time_point_idx])
        obs_times.extend(batch_res["time"][mask_line, line_idx])
        obs_events.extend(batch_res["event"][mask_line, line_idx])

    preds_surv = np.array(preds_surv)  # of size ( valid_lines, n_intervals), the n
    obs_times = np.array(obs_times)
    obs_events = np.array(obs_events)

    # Bin predictions
    bins = np.linspace(0, 1, n_bins + 1)  # discreatize the [0,1] interval into n_bins
    # bins = np.quantile(preds_surv, np.linspace(0, 1, n_bins+1)) #discreatize the [0,1] interval into n_bins
    bin_indices = np.digitize(preds_surv, bins) - 1  # which bin each pred falls into
    calib_pred = []
    calib_obs = []
    calib_pred_bound = []
    calib_obs_bound = []

    for i in range(n_bins):
        idx = bin_indices == i
        if idx.sum() < 10:
            # print(f"Bin {i}: n={idx.sum()}")
            continue
        avg_pred = np.median(preds_surv[idx])
        upper_bound = np.quantile(preds_surv[idx], 0.95)
        lower_bound = np.quantile(preds_surv[idx], 0.05)

        # Compute KM survival at time_val for this bin
        time, survival_prob, conf_int = kaplan_meier_estimator(
            obs_events[idx].astype(bool), obs_times[idx], conf_type="log-log"
        )

        # Handle case where time_val is outside the observed range
        if time_val > time[-1]:
            # print("Extrapolating beyond observed times.")
            # print(time_val, time[-1])
            # Extrapolate: use last known survival probability
            est = survival_prob[-1]
            lower_bound = conf_int[0, -1]
            upper_bound = conf_int[1, -1]
        elif time_val < time[0]:
            # print("Time point before observed times.")
            # Before any events: survival is 1.0
            est = 1.0
        else:
            # print("Within observed times.")
            # Within range: use step function
            km_func = StepFunction(time, survival_prob)
            est = km_func(time_val)
            # print(len(time), conf_int.shape)
            lower_bound = np.interp(time_val, time, conf_int[0, :])
            upper_bound = np.interp(time_val, time, conf_int[1, :])

        calib_obs.append(est)
        calib_pred.append(avg_pred)
        calib_pred_bound.append((lower_bound, upper_bound))
        calib_obs_bound.append((lower_bound, upper_bound))
    return (
        np.array(calib_pred),
        np.array(calib_obs),
        np.array(calib_pred_bound),
        np.array(calib_obs_bound),
    )

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=N_LINES, figsize=(5 * N_LINES, 4))
for line_idx in range(N_LINES):
    n_samples = 10
    sample_times = np.random.choice(np.arange(1, len(model.interval_bounds[line_idx]) - 1), size=n_samples, replace=False)
    sample_times.sort()
    for i in sample_times:

        # print(f"Line {l+1}, Time Point Index {i}")
        calib_pred, calib_obs, calib_pred_bound, calib_obs_bound = plot_calibration(
            line_idx, time_point_idx=i, n_bins=10
        )
        # print(calib_pred_bound)
        ax[line_idx].plot(
            calib_pred,
            calib_obs,
            ".-",
            label=f"t={model.interval_bounds[line_idx][i + 1].item():.1f}",
        )
        # ax[line_idx].errorbar(
        # calib_pred, calib_obs,
        # xerr=[np.abs(calib_pred - calib_pred_bound[:,0]), np.abs(calib_pred_bound[:,1] - calib_pred)],
        # label=f"t={model.interval_bounds[i+1].item():.1f}"
        # )

        ax[line_idx].set_xlabel("Predicted Survival")
    ax[line_idx].plot([0, 1], [0, 1], "k--")
    ax[line_idx].set_ylabel("Observed Survival (KM)")
    ax[line_idx].legend()
    ax[line_idx].set_title(f"Calibration Plot - Line {line_idx + 1}")
    ax[line_idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Adversarial unconfounding
Check if the learned representation is not predictive of the treatment

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

probabilities = {}
aacuracy = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        h, c, p = model._init_lstm_states(X_static, XPd.device)
        for line in range(XPd.shape[1]):
            _, h, c, p = model.lstm(XPd[:, line, :], (h, c, p))

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].numpy()
            y = treatment_indices[mask_line, line].numpy()
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_train, y_train)
            rfc.predict_proba(X_test)
            acc = rfc.score(X_test, y_test)
            aacuracy[line] = acc
            probabilities[line] = rfc.predict_proba(X_test)

print("\n--- Treatment Assignment Accuracy ---")
for line in range(N_LINES):
    acc = aacuracy.get(line, None)
    if acc is not None:
        print(f"Line {line + 1}: Treatment Assignment Accuracy = {acc:.4f}")
    else:
        print(f"Line {line + 1}: No valid data for treatment assignment accuracy.")

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns


def plot_umap_deconfounding(
    hidden_states, treatment_labels, title="UMAP Projection of LSTM Hidden States"
):
    """
    hidden_states: numpy array of shape (n_samples, hidden_dim)
    treatment_labels: 1D array of treatment categories
    """
    # 1. Initialize and fit UMAP
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",  # Cosine is often better for high-dim hidden states
    )
    embedding = reducer.fit_transform(hidden_states)

    # 2. Prepare DataFrame for plotting
    df = pd.DataFrame(embedding, columns=["UMAP 1", "UMAP 2"])
    df["Treatment"] = treatment_labels

    # print(df['Treatment'].value_counts())
    # 3. Create the plot
    plt.figure(figsize=(5, 5))
    sns.scatterplot(
        data=df,
        x="UMAP 1",
        y="UMAP 2",
        hue="Treatment",
        palette="viridis",
        alpha=0.6,
        s=15,
    )

    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

probabilities = {}
accuracies = {}
auc_scores = {}
dummy_auc_scores = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        h, c, p = model._init_lstm_states(X_static, XPd.device)

        for line in range(XPd.shape[1]):
            _, h, c, p = model.lstm(XPd[:, line, :], (h, c, p))

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].cpu().numpy()
            y = treatment_indices[mask_line, line].cpu().numpy()
            # print(np.unique(y, return_counts=True))
            # plot_umap_deconfounding(X, y, title=f"UMAP Projection - Line {line + 1}")

            unique_classes, counts = np.unique(y, return_counts=True)
            valid_classes = unique_classes[counts >= 5]
            valid_indices = np.isin(y, valid_classes)
            X = X[valid_indices]
            y = y[valid_indices]

            # print(X.shape, y.shape)
            if len(np.unique(y)) < 2:
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_train, y_train)
            acc = rfc.score(X_test, y_test)
            rfc_probs = rfc.predict_proba(X_test)

            dummy = DummyClassifier(strategy="stratified", random_state=42)
            dummy.fit(X_train, y_train)
            dummy_probs = dummy.predict_proba(X_test)

            # print(f"random forest probs:{rfc_probs.shape}, y_test.shape:{y_test.shape}")
            # print(f"dummy dummy_probs.shape:{dummy_probs.shape}, y_test.shape:{y_test.shape}")

            rfc_auc = roc_auc_score(
                y_test,
                rfc_probs,
                multi_class="ovr",
                average="macro",
                labels=rfc.classes_,
            )
            dummy_auc = roc_auc_score(
                y_test,
                dummy_probs,
                multi_class="ovr",
                average="macro",
                labels=dummy.classes_,
            )

            # Store results
            accuracies[line] = acc
            auc_scores[line] = rfc_auc
            dummy_auc_scores[line] = dummy_auc
            probabilities[line] = rfc_probs

# --- Print Results Table ---
print(f"\n{'Line':<10} | {'RFC Accuracy':<15} | {'RFC AUC':<10} | {'Dummy AUC':<10}")
print("-" * 55)
for line in range(N_LINES):
    if line in accuracies:
        print(
            f"Line {line + 1:<5} | {accuracies[line]:.4f}         | {auc_scores[line]:.4f}     | {dummy_auc_scores[line]:.4f}"
        )
    else:
        print(f"Line {line + 1:<5} | No valid data.")

## Treatment Policy

In [ ]:
# Regret Analysis
regrets = {l: [] for l in range(N_LINES)}
interval_bounds = model.interval_bounds[0]
print("Computing Regret...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Regret"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch

        hazards = torch.sigmoid(
            model(XPd, X_static)[0]
        )  # (batch, n_lines, n_treatments, n_intervals)
        survival = torch.cumprod(
            1 - hazards, dim=3
        )  # (batch, n_lines, n_treatments, n_intervals)

        dt = interval_bounds[1] - interval_bounds[0]  # Approx
        utility = survival.sum(dim=3) * dt  # (batch, n_lines, n_treatments)

        max_utility, _ = utility.max(dim=2)

        idx_expanded = treatment_indices.unsqueeze(-1)  # (batch, n_lines, 1)
        obs_utility = torch.gather(utility, 2, idx_expanded).squeeze(2)

        regret = max_utility - obs_utility

        for line in range(N_LINES):
            valid = mask[:, line].bool()
            if valid.any():
                regrets[line].extend(regret[valid, line].cpu().numpy())

# Histograms
plt.figure(figsize=(12, 4))
for line in range(N_LINES):
    plt.subplot(1, N_LINES, line + 1)
    plt.hist(regrets[line], bins=30, alpha=0.7)
    plt.title(f"Regret Dist - Line {line + 1}")
    plt.xlabel("Regret (RMST Difference)")
plt.tight_layout()
plt.show()

## Virtual clinical trial

In [ ]:
concordant_group = {l: {"time": [], "event": []} for l in range(N_LINES)}
discordant_group = {l: {"time": [], "event": []} for l in range(N_LINES)}

print("Running Virtual Clinical Trial (Concordance Analysis)...")

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Policy"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch

        hazards = torch.sigmoid(model(XPd, X_static)[0])
        survival_curves = torch.cumprod(1 - hazards, dim=3)

        dt = interval_bounds[1] - interval_bounds[0]
        rmst = survival_curves.sum(dim=3) * dt  # (batch, n_lines, n_treatments)

        recommended_treatment = rmst.argmax(dim=2)  # (batch, n_lines)
        observed_treatment = treatment_indices  # (batch, n_lines)

        for line in range(N_LINES):
            valid_mask = mask[:, line].bool()
            if not valid_mask.any():
                continue

            rec = recommended_treatment[valid_mask, line].cpu().numpy()
            obs = observed_treatment[valid_mask, line].cpu().numpy()

            real_t = time[valid_mask, line].cpu().numpy()
            real_e = event[valid_mask, line].cpu().numpy()

            matches = rec == obs

            concordant_group[line]["time"].extend(real_t[matches])
            concordant_group[line]["event"].extend(real_e[matches])

            discordant_group[line]["time"].extend(real_t[~matches])
            discordant_group[line]["event"].extend(real_e[~matches])

fig, axes = plt.subplots(1, N_LINES, figsize=(6 * N_LINES, 5))
if N_LINES == 1:
    axes = [axes]

for line in range(N_LINES):
    ax = axes[line]

    kmf_concordant = KaplanMeierFitter()
    kmf_discordant = KaplanMeierFitter()

    T_conc, E_conc = concordant_group[line]["time"], concordant_group[line]["event"]
    T_disc, E_disc = discordant_group[line]["time"], discordant_group[line]["event"]

    if len(T_conc) > 0:
        kmf_concordant.fit(
            T_conc, event_observed=E_conc, label=f"Matched Policy (n={len(T_conc)})"
        )
        kmf_concordant.plot_survival_function(ax=ax, ci_show=True, color="green")

    if len(T_disc) > 0:
        kmf_discordant.fit(
            T_disc, event_observed=E_disc, label=f"Differed Policy (n={len(T_disc)})"
        )
        kmf_discordant.plot_survival_function(ax=ax, ci_show=True, color="red")

    if len(T_conc) > 0 and len(T_disc) > 0:
        results = logrank_test(
            T_conc, T_disc, event_observed_A=E_conc, event_observed_B=E_disc
        )
        p_value = results.p_value

        t_max = min(np.max(T_conc), np.max(T_disc))
        rmst_conc = kmf_concordant.predict(range(int(t_max))).sum()
        rmst_disc = kmf_discordant.predict(range(int(t_max))).sum()
        delta_rmst = rmst_conc - rmst_disc

        title = (
            f"Line {line + 1}\nLog-Rank p={p_value}\nRMST Diff: {delta_rmst:.2f} months"
        )
    else:
        title = f"Line {line + 1} (Insufficient Data)"

    ax.set_title(title)
    ax.set_xlabel("Observed Time")
    ax.set_ylabel("Survival Probability")
    ax.grid(True, alpha=0.3)

fig.suptitle("Model Recommendation vs. Actual Treatment")
plt.tight_layout()
plt.show()

## Treatment recommendation analysis

# 🚧🚧🚧🚧🚧🚧🚧

In [ ]:
# from sksurv.metrics import brier_score, cumulative_dynamic_auc
# from sksurv.nonparametric import kaplan_meier_estimator
# import numpy as np
# import matplotlib.pyplot as plt

# def plot_calibration_improved(preds_surv, obs_times, obs_events, time_point,
#                                n_bins=10, strategy='quantile'):
#     """
#     Better calibration plot with more robust binning and error bars.

#     Parameters:
#     - strategy: 'uniform' (equal width) or 'quantile' (equal sample size per bin)
#     """

#     # Remove invalid predictions
#     valid_mask = ~np.isnan(preds_surv) & (preds_surv >= 0) & (preds_surv <= 1)
#     preds_surv = preds_surv[valid_mask]
#     obs_times = obs_times[valid_mask]
#     obs_events = obs_events[valid_mask]

#     if len(preds_surv) < n_bins * 5:
#         print(f"Warning: Only {len(preds_surv)} samples, reducing bins")
#         n_bins = max(3, len(preds_surv) // 10)

#     # Better binning strategy
#     if strategy == 'quantile':
#         # Equal number of samples per bin
#         bin_edges = np.percentile(preds_surv, np.linspace(0, 100, n_bins + 1))
#         bin_edges[-1] += 1e-8  # Ensure last bin includes max value
#     else:
#         # Equal width bins
#         bin_edges = np.linspace(0, 1, n_bins + 1)

#     bin_indices = np.digitize(preds_surv, bin_edges) - 1
#     bin_indices = np.clip(bin_indices, 0, n_bins - 1)  # Handle edge cases

#     calib_pred = []
#     calib_obs = []
#     calib_counts = []
#     calib_se = []  # Standard errors for confidence intervals

#     for i in range(n_bins):
#         idx = bin_indices == i
#         count = idx.sum()

#         if count < 5:  # Skip bins with too few samples
#             continue

#         # Average prediction in bin
#         avg_pred = preds_surv[idx].mean()

#         # Kaplan-Meier estimate at time_point
#         try:
#             time, survival_prob = kaplan_meier_estimator(
#                 obs_events[idx].astype(bool),
#                 obs_times[idx]
#             )

#             # Interpolate at time_point
#             if time_point <= time[0]:
#                 obs_surv = 1.0
#                 se = 0.0
#             elif time_point >= time[-1]:
#                 obs_surv = survival_prob[-1]
#                 # Greenwood's formula for KM variance (simplified)
#                 se = np.sqrt(survival_prob[-1] * (1 - survival_prob[-1]) / count)
#             else:
#                 # Find surrounding time points
#                 idx_after = np.searchsorted(time, time_point)
#                 obs_surv = survival_prob[idx_after - 1]  # Step function
#                 se = np.sqrt(obs_surv * (1 - obs_surv) / count)

#             calib_obs.append(obs_surv)
#             calib_pred.append(avg_pred)
#             calib_counts.append(count)
#             calib_se.append(se)

#         except Exception as e:
#             print(f"Skipping bin {i}: {e}")
#             continue

#     # Convert to arrays
#     calib_pred = np.array(calib_pred)
#     calib_obs = np.array(calib_obs)
#     calib_se = np.array(calib_se)

#     # Plot with error bars
#     plt.figure(figsize=(7, 7))

#     # Perfect calibration
#     plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label="Perfect Calibration", alpha=0.7)

#     # Calibration curve with confidence intervals
#     plt.errorbar(calib_pred, calib_obs, yerr=1.96*calib_se,
#                  fmt='o-', capsize=5, capthick=2,
#                  label=f"Observed (n={len(preds_surv)})")

#     # Add sample size annotations
#     for pred, obs, count in zip(calib_pred, calib_obs, calib_counts):
#         plt.annotate(f'n={count}', (pred, obs),
#                     textcoords="offset points", xytext=(0,10),
#                     ha='center', fontsize=8, alpha=0.6)

#     plt.xlabel("Predicted Survival Probability", fontsize=12)
#     plt.ylabel("Observed Survival Probability (KM)", fontsize=12)
#     plt.title(f"Calibration Plot at t={time_point:.1f}", fontsize=14)
#     plt.legend(loc='best')
#     plt.grid(True, alpha=0.3)
#     plt.xlim(-0.05, 1.05)
#     plt.ylim(-0.05, 1.05)
#     plt.tight_layout()

#     # Calculate calibration metrics
#     ece = np.sum(calib_counts * np.abs(calib_pred - calib_obs)) / sum(calib_counts)
#     print(f"Expected Calibration Error (ECE): {ece:.4f}")

#     return calib_pred, calib_obs, ece

In [ ]:
# from lifelines import KaplanMeierFitter
# from scipy.stats import kstest

# def d_calibration(preds_surv, obs_times, obs_events, time_point):
#     """
#     D-calibration: Tests if distribution of predictions matches observed.
#     Uses probability integral transform (PIT).
#     """

#     # Compute PIT values
#     pit_values = []
#     for pred, time, event in zip(preds_surv, obs_times, obs_events):
#         if time < time_point:
#             # Event occurred before time_point
#             pit = pred if event else 1.0
#         else:
#             # Censored or survived past time_point
#             pit = pred
#         pit_values.append(pit)

#     pit_values = np.array(pit_values)

#     # If well-calibrated, PIT should be uniform [0,1]
#     ks_stat, p_value = kstest(pit_values, 'uniform')

#     print(f"KS test: statistic={ks_stat:.4f}, p-value={p_value:.4f}")

#     # Plot PIT histogram
#     plt.figure(figsize=(8, 5))
#     plt.hist(pit_values, bins=20, density=True, alpha=0.7, edgecolor='black')
#     plt.axhline(1.0, color='red', linestyle='--', label='Perfect calibration (uniform)')
#     plt.xlabel('PIT Values')
#     plt.ylabel('Density')
#     plt.title(f'D-Calibration at t={time_point:.1f}')
#     plt.legend()
#     plt.tight_layout()

#     return ks_stat, p_value

In [ ]:
# # Evaluate all treatment lines and time points
# N_LINES = 3  # Adjust to your number of lines

# for line_idx in range(N_LINES):
#     print(f"\n{'='*60}")
#     print(f"Treatment Line {line_idx + 1}")
#     print(f"{'='*60}")

#     # Create subplot figure for this line
#     n_timepoints = len(model.interval_bounds) - 2
#     fig, axes = plt.subplots(1, min(3, n_timepoints), figsize=(15, 5))
#     if n_timepoints == 1:
#         axes = [axes]

#     for tp_idx in range(1, min(4, len(model.interval_bounds)-1)):
#         # Gather data for this line and time point
#         preds_surv = []
#         obs_times = []
#         obs_events = []
#         time_val = model.interval_bounds[tp_idx].item()

#         for batch_res in all_predictions:
#             mask_line = batch_res["mask"][:, line_idx].astype(bool)
#             if not mask_line.any():
#                 continue
#             haz_logits = batch_res["hazards_logit"][mask_line, line_idx, :]
#             haz = 1 / (1 + np.exp(-haz_logits))
#             surv = np.cumprod(1 - haz, axis=1)
#             preds_surv.extend(surv[:, tp_idx])
#             obs_times.extend(batch_res["time"][mask_line, line_idx])
#             obs_events.extend(batch_res["event"][mask_line, line_idx])

#         preds_surv = np.array(preds_surv)
#         obs_times = np.array(obs_times)
#         obs_events = np.array(obs_events)

#         print(f"\nTime point {tp_idx} (t={time_val:.2f}):")

#         # Run calibration
#         calib_pred, calib_obs, ece = plot_calibration_improved(
#             preds_surv, obs_times, obs_events, time_val, n_bins=10
#         )

#         # Run D-calibration
#         ks_stat, p_value = d_calibration(
#             preds_surv, obs_times, obs_events, time_val
#         )

#     plt.show()